# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 01.01 · Adquisición incremental de subtítulos

Reutiliza transcripciones canónicas y cachés por `video_id`; solo consulta YouTube para candidatos nuevos y nunca descarga audio o video.

La adquisición nueva usa `yt-dlp` para localizar pistas de subtítulos [1]. Las transcripciones automáticas se conservan como insumo imperfecto, no como verdad textual, porque se han documentado sesgos de dialecto y género en el subtitulado automático de YouTube [2]. Toda ampliación debe respetar los términos de la plataforma [3] y la evaluación ética contextual recomendada para investigación en Internet [4]. La reutilización de cachés y la selección de candidatos son decisiones locales registradas en manifiestos. El modo dirigido adapta principios de balance para aprendizaje activo y clasificación multietiqueta de cola larga [5] [6]: calcula soporte por videos únicos solo en `train+validation`, excluye `test`, pondera los déficits, estima rendimiento histórico por canal, expande canales desde consultas temáticas y materializa una cohorte aislada mediante *round-robin*. Si no hay etiquetas previas utilizables, asigna 25% a cada daño. La fórmula, los umbrales, las cuotas, el fallback y el cortacircuito HTTP 429 son decisiones operativas propias; `target_category` registra el motivo de selección y nunca constituye una etiqueta. Este muestreo enriquecido no permite estimar prevalencias en YouTube ni en el Perú.

**Contrato v2.1:** `SEGURO` + cuatro daños entrenados, incluida `ATAQUE_POR_GENERO_IDENTIDAD`. `SEGURO` es excluyente; los daños son multietiqueta. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')


## Preflight

In [ ]:
from moderacion_peru.artifacts import artifact_status
show_result('Disponibilidad de artefactos', artifact_status(ROOT), tone='neutral')

## Parámetros editables

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONTROLES DEL SCRAPING: edite únicamente este bloque
# ══════════════════════════════════════════════════════════════════════════════
DISCOVER_NEW = True           # True: consulta canales/búsquedas y guarda candidatos
FETCH_NEW = True              # True: obtiene subtítulos solo de videos aún no procesados
DISCOVERY_MODE = "directed"   # "seed", "directed" o "both"

MAX_NEW_VIDEOS = 2000         # máximo de llamadas nuevas para obtener subtítulos
MAX_VIDEOS_PER_CHANNEL = 75   # candidatos recientes inspeccionados por canal
MAX_RESULTS_PER_QUERY = 20    # candidatos inspeccionados por consulta dirigida
MAX_DIRECTED_CANDIDATES = 500 # tamaño máximo de la cohorte dirigida vigente
MAX_DIRECTED_SEED_CHANNELS = 12
MAX_EXPANDED_CHANNELS = 12    # canales nuevos inferidos desde búsquedas temáticas
MAX_VIDEOS_PER_EXPANDED_CHANNEL = 20

SUBTITLE_LANGUAGES = ("es-PE", "es-419", "es")
YT_RETRIES = 3
YT_SLEEP_MIN_SECONDS = 1.0
YT_SLEEP_MAX_SECONDS = 3.0
STOP_ON_VIDEO_ERROR = False   # False: registra el fallo y continúa con el siguiente

# Reinicio destructivo recuperable: deje vacío normalmente. Para archivar los
# artefactos activos y reconstruir el dataset de videos desde cero, descomente:
RESET_VIDEO_DATASET = ""
# RESET_VIDEO_DATASET = "ARCHIVAR_Y_REINICIAR_DATASET_VIDEOS"

if DISCOVERY_MODE not in {"seed", "directed", "both"}:
    raise ValueError("DISCOVERY_MODE debe ser seed, directed o both")
if (MAX_NEW_VIDEOS < 0 or MAX_VIDEOS_PER_CHANNEL < 1 or MAX_RESULTS_PER_QUERY < 1
        or MAX_DIRECTED_CANDIDATES < 1 or MAX_DIRECTED_SEED_CHANNELS < 1
        or MAX_EXPANDED_CHANNELS < 0 or MAX_VIDEOS_PER_EXPANDED_CHANNEL < 1):
    raise ValueError("Los límites de videos deben ser válidos")
if YT_SLEEP_MIN_SECONDS < 0 or YT_SLEEP_MAX_SECONDS < YT_SLEEP_MIN_SECONDS:
    raise ValueError("El intervalo de espera de yt-dlp no es válido")

show_summary('Configuración del scraping', {
    "discover_new": DISCOVER_NEW,
    "fetch_new": FETCH_NEW,
    "discovery_mode": DISCOVERY_MODE,
    "max_new_videos": MAX_NEW_VIDEOS,
    "max_videos_per_channel": MAX_VIDEOS_PER_CHANNEL,
    "max_results_per_query": MAX_RESULTS_PER_QUERY,
    "max_directed_candidates": MAX_DIRECTED_CANDIDATES,
    "max_directed_seed_channels": MAX_DIRECTED_SEED_CHANNELS,
    "max_expanded_channels": MAX_EXPANDED_CHANNELS,
    "subtitle_languages": SUBTITLE_LANGUAGES,
    "reset_video_dataset_armed": bool(RESET_VIDEO_DATASET),
}, tone='neutral')

if RESET_VIDEO_DATASET:
    from moderacion_peru.acquisition import reset_active_video_dataset
    reset_result = reset_active_video_dataset(ROOT, RESET_VIDEO_DATASET)
    show_result('Dataset activo archivado; reconstrucción desde cero habilitada', reset_result, tone='warning')


## Canales y consultas

In [ ]:
# Canales semilla recuperados del cuaderno histórico; puede añadir, quitar o editar filas.
# `categoria_fuente` describe el dominio/registro del canal, no una etiqueta de daño.
_SEED_ROWS = [
    ("Marco Sifuentes / Ocram", "politica_analisis", "https://www.youtube.com/@canalYAAAAA"),
    ("El diario de Curwen", "politica_opinion", "https://www.youtube.com/@curwen"),
    ("Sin Guion con Rosa María Palacios", "politica_periodismo", "https://www.youtube.com/@singuionlr"),
    ("RPP Noticias", "politica_actualidad", "https://www.youtube.com/@RPPNoticias"),
    ("Exitosa Noticias", "politica_actualidad", "https://www.youtube.com/@exitosape"),
    ("Willax Television", "politica_opinion", "https://www.youtube.com/@WillaxTV"),
    ("Canal N", "politica_actualidad", "https://www.youtube.com/@canaln"),
    ("ATV Noticias", "politica_actualidad", "https://www.youtube.com/@ATVNoticias"),
    ("Latina Noticias", "politica_actualidad", "https://www.youtube.com/@latinanoticias"),
    ("Panamericana Noticias", "politica_actualidad", "https://www.youtube.com/@Panamericana-Noticias"),
    ("Hablando Huevadas", "humor_streaming", "https://www.youtube.com/@HablandoHuevadasOficial"),
    ("Todo Good", "streaming_opinion", "https://www.youtube.com/@todogoodpe"),
    ("Goblinciano", "streaming_opinion", "https://www.youtube.com/@Goblinciano"),
    ("El Cacas", "humor_streaming", "https://www.youtube.com/@ElCacas"),
    ("Negro Fuertes", "humor_comedia", "https://www.youtube.com/@NegroFuertes"),
    ("Jason Qqq", "humor_streaming", "https://www.youtube.com/@JasonQqqOficial"),
    ("La Cotorrisa Perú", "humor_podcast", "https://www.youtube.com/@LaCotorrisaPeru"),
    ("Magaly TV La Firme", "farandula", "https://www.youtube.com/@MagalyTVLaFirmeATV"),
    ("Amor y Fuego", "farandula", "https://www.youtube.com/@AmoryFuego"),
    ("América Hoy", "farandula", "https://www.youtube.com/@americahoytv"),
    ("Instarándula", "farandula_digital", "https://www.youtube.com/@Instarandula"),
    ("El Popular", "farandula_digital", "https://www.youtube.com/@ElPopularPeru"),
    ("Nico Moschella", "deportes_informal", "https://www.youtube.com/@NicoMoschella"),
    ("Líbero Deportes", "deportes", "https://www.youtube.com/@DiarioLiberoOficial"),
    ("Depor", "deportes", "https://www.youtube.com/@DeporPeru"),
    ("Misias pero viajeras", "viajes", "https://www.youtube.com/c/Misiasperoviajeras"),
    ("Buen Viaje", "viajes", "https://www.youtube.com/c/BuenViajePe"),
    ("Viaja y Prueba", "viajes_gastronomia", "https://www.youtube.com/@ViajayPrueba"),
    ("Cocinando con la Patty", "gastronomia", "https://www.youtube.com/@CocinandoConLaPatty"),
    ("Arde Troya con Juliana Oxenford", "politica_analisis", "https://www.youtube.com/@ardetroyalr"),
    ("Panorama", "politica_periodismo", "https://www.youtube.com/@PanoramaPTV"),
    ("Juanito y Richard", "humor_comedia", "https://www.youtube.com/@JuanitoyRichard"),
    ("Nada Espacial", "humor_podcast", "https://www.youtube.com/@nadaespacialpodcast"),
    ("L1MAX", "deportes_informal", "https://www.youtube.com/@L1MAX_"),
    ("Cocina Cajamarquina", "gastronomia", "https://www.youtube.com/@cocinacajamarquina"),
    ("Tío Lenguado y Descocaos", "viajes_gastronomia", "https://www.youtube.com/@tiolenguado"),
]
SEED_CHANNELS = [
    {"name": name, "categoria_fuente": category, "url": url, "sampling_mode": "seed"}
    for name, category, url in _SEED_ROWS
]

# Ampliación dirigida: las cuotas son máximos por canal, nunca prevalencias esperadas.
DIRECTED_CHANNEL_CATALOG = [
    {"name": "Hablando Huevadas", "url": "https://www.youtube.com/@HablandoHuevadasOficial", "quota": 70, "target_category": "CONTENIDO_SEXUAL|ATAQUE_POR_GENERO_IDENTIDAD|ACOSO_AMENAZA"},
    {"name": "Goblinciano", "url": "https://www.youtube.com/@Goblinciano", "quota": 85, "target_category": "RACISMO_DISCRIMINACION|ACOSO_AMENAZA"},
    {"name": "Juanito y Richard", "url": "https://www.youtube.com/@JuanitoyRichard", "quota": 85, "target_category": "RACISMO_DISCRIMINACION|ACOSO_AMENAZA"},
    {"name": "Arde Troya con Juliana Oxenford", "url": "https://www.youtube.com/@ardetroyalr", "quota": 55, "target_category": "ACOSO_AMENAZA"},
    {"name": "Todo Good", "url": "https://www.youtube.com/@todogoodpe", "quota": 40, "target_category": "ACOSO_AMENAZA"},
    {"name": "Magaly TV La Firme", "url": "https://www.youtube.com/@MagalyTVLaFirmeATV", "quota": 35, "target_category": "ACOSO_AMENAZA|CONTENIDO_SEXUAL"},
]

SEED_SEARCH_QUERIES = [
    "noticias política Perú canal YouTube",
    "periodismo opinión Perú YouTube",
    "humor streaming Perú lenguaje coloquial",
    "farándula espectáculos Perú",
    "deportes peruanos comentarios YouTube",
]
DIRECTED_QUERY_CATALOG = [
    {"query": "insultos racistas discriminación Perú denuncia", "target_category": "RACISMO_DISCRIMINACION"},
    {"query": "discriminación regional clasismo racial Perú", "target_category": "RACISMO_DISCRIMINACION"},
    {"query": "ataque machista misoginia Perú denuncia", "target_category": "ATAQUE_POR_GENERO_IDENTIDAD"},
    {"query": "ataque homofóbico transfóbico Perú denuncia", "target_category": "ATAQUE_POR_GENERO_IDENTIDAD"},
    {"query": "amenaza de muerte denuncia Perú", "target_category": "ACOSO_AMENAZA"},
    {"query": "extorsionadores amenazan audio Perú", "target_category": "ACOSO_AMENAZA"},
    {"query": "acoso sexual denuncia televisión peruana", "target_category": "CONTENIDO_SEXUAL|ACOSO_AMENAZA"},
    {"query": "cosificación contenido sexual explícito televisión Perú", "target_category": "CONTENIDO_SEXUAL"},
]


## Reutilización y plan dirigido

El soporte se mide por `video_id` único para no favorecer videos largos con más chunks. La ponderación por déficit y el reparto multietiqueta se inspiran en problemas de aprendizaje activo desbalanceado y distribuciones de cola larga [5] [6]. Su implementación exacta es local: `test` permanece congelado, el fallback sin historia reparte las cuatro categorías por igual y las categorías objetivo solo orientan la adquisición.

In [ ]:
from moderacion_peru.acquisition import (
    VIDEO_DATASET_RESET_MARKER,
    bootstrap_canonical_from_existing,
    build_directed_sampling_plan,
    discover_existing_transcript_sources,
    load_candidates,
    merge_candidates,
    select_directed_search_queries,
    select_directed_seed_channels,
)
from moderacion_peru.io import read_jsonl
from moderacion_peru.taxonomy import load_taxonomy

CANONICAL = ROOT/'datos/raw/transcripts_raw.jsonl'
CACHE = ROOT/'datos/raw/transcripts_cache'
DIRECTED_DATASET = ROOT/'datos/model_ready/v2/dataset_5_salidas.jsonl'
rebuild_from_zero = (ROOT/VIDEO_DATASET_RESET_MARKER).exists()
sources = [] if rebuild_from_zero else discover_existing_transcript_sources(ROOT, canonical_path=CANONICAL)
reuse_stats = bootstrap_canonical_from_existing(sources, CANONICAL)
reuse_stats['historical_bootstrap_disabled'] = rebuild_from_zero
show_result('Reutilización de snapshots', reuse_stats, tone='warning' if rebuild_from_zero else 'success')

taxonomy = load_taxonomy(ROOT/'config/taxonomia_v2.json')
directed_plan = None
DIRECTED_CHANNELS = []
DIRECTED_SEARCH_QUERIES = []
if DISCOVERY_MODE in {'directed', 'both'}:
    directed_plan = build_directed_sampling_plan(
        read_jsonl(DIRECTED_DATASET) if DIRECTED_DATASET.exists() else [],
        read_jsonl(CANONICAL) if CANONICAL.exists() else [],
        damage_labels=taxonomy.damage_labels,
    )
    DIRECTED_CHANNELS = select_directed_seed_channels(
        directed_plan,
        DIRECTED_CHANNEL_CATALOG,
        max_channels=MAX_DIRECTED_SEED_CHANNELS,
    )
    DIRECTED_SEARCH_QUERIES = select_directed_search_queries(
        directed_plan,
        DIRECTED_QUERY_CATALOG,
        max_queries=len(DIRECTED_QUERY_CATALOG),
        max_results_per_query=MAX_RESULTS_PER_QUERY,
    )
    show_summary('Plan de ampliación dirigida', {
        'estrategia': directed_plan['strategy'],
        'soporte_videos_train_validation': directed_plan['support_videos'],
        'déficit_videos': directed_plan['deficit_videos'],
        'pesos': {key: round(value, 4) for key, value in directed_plan['weights'].items()},
        'canales_semilla': len(DIRECTED_CHANNELS),
        'consultas_temáticas': len(DIRECTED_SEARCH_QUERIES),
    }, tone='warning' if directed_plan['strategy'] == 'fallback_equal' else 'success')

CHANNEL_SOURCES = (
    SEED_CHANNELS if DISCOVERY_MODE == 'seed'
    else DIRECTED_CHANNELS if DISCOVERY_MODE == 'directed'
    else SEED_CHANNELS + DIRECTED_CHANNELS
)
SEARCH_QUERIES = (
    SEED_SEARCH_QUERIES if DISCOVERY_MODE == 'seed'
    else DIRECTED_SEARCH_QUERIES if DISCOVERY_MODE == 'directed'
    else SEED_SEARCH_QUERIES + DIRECTED_SEARCH_QUERIES
)
show_summary('Fuentes seleccionadas', {
    'modo': DISCOVERY_MODE,
    'canales': len(CHANNEL_SOURCES),
    'consultas': len(SEARCH_QUERIES),
}, tone='neutral')


## Descubrimiento general o ampliación dirigida

In [ ]:
from collections import Counter
from tqdm.auto import tqdm
from moderacion_peru.acquisition import (
    discover_youtube_candidates,
    expand_directed_channel_sources,
    processed_video_ids,
    select_directed_candidates,
)
from moderacion_peru.io import append_jsonl_once, write_json_atomic, write_jsonl_atomic

DISCOVERED_PATH = ROOT/'datos/raw/video_candidates.jsonl'
DISCOVERY_FAILURES_PATH = ROOT/'datos/raw/fallos_descubrimiento_ultima_ejecucion.json'
DIRECTED_SELECTION_PATH = ROOT/'datos/raw/directed_candidates_latest.jsonl'
DIRECTED_PLAN_PATH = ROOT/'datos/raw/manifests/directed_plan_latest.json'
discovered = []
directed_selection = []
expanded_channels = []
if DISCOVER_NEW:
    source_outcomes = Counter()
    source_total = len(CHANNEL_SOURCES) + len(SEARCH_QUERIES)
    source_progress = tqdm(total=source_total, desc='Descubriendo fuentes', unit='fuente')

    def report_discovery(event):
        source_outcomes[event['status']] += 1
        source_progress.update(1)
        source_progress.set_postfix(
            correctas=source_outcomes['ok'],
            fallidas=source_outcomes['failed'],
            candidatos=event['candidates_unique'],
        )

    try:
        discovered, discovery_failures = discover_youtube_candidates(
            CHANNEL_SOURCES,
            SEARCH_QUERIES,
            max_videos_per_channel=MAX_VIDEOS_PER_CHANNEL,
            max_results_per_query=MAX_RESULTS_PER_QUERY,
            retries=YT_RETRIES,
            sleep_min_seconds=YT_SLEEP_MIN_SECONDS,
            sleep_max_seconds=YT_SLEEP_MAX_SECONDS,
            progress_callback=report_discovery,
        )
        if directed_plan is not None:
            expanded_channels = expand_directed_channel_sources(
                discovered,
                directed_plan,
                known_channel_ids=[source.get('channel_id') for source in DIRECTED_CHANNELS],
                max_channels=MAX_EXPANDED_CHANNELS,
                videos_per_channel=MAX_VIDEOS_PER_EXPANDED_CHANNEL,
            )
            if expanded_channels:
                source_total += len(expanded_channels)
                source_progress.total = source_total
                source_progress.refresh()
                expanded_candidates, expanded_failures = discover_youtube_candidates(
                    expanded_channels,
                    (),
                    max_videos_per_channel=MAX_VIDEOS_PER_EXPANDED_CHANNEL,
                    max_results_per_query=MAX_RESULTS_PER_QUERY,
                    retries=YT_RETRIES,
                    sleep_min_seconds=YT_SLEEP_MIN_SECONDS,
                    sleep_max_seconds=YT_SLEEP_MAX_SECONDS,
                    progress_callback=report_discovery,
                )
                discovered = merge_candidates(discovered, expanded_candidates)
                discovery_failures.extend(expanded_failures)
            directed_pool = [
                candidate for candidate in discovered
                if candidate.get('sampling_mode') == 'directed'
            ]
            directed_selection = select_directed_candidates(
                directed_pool,
                processed_video_ids(CANONICAL),
                directed_plan,
                max_candidates=MAX_DIRECTED_CANDIDATES,
            )
            write_jsonl_atomic(DIRECTED_SELECTION_PATH, directed_selection)
            write_json_atomic(DIRECTED_PLAN_PATH, {
                **directed_plan,
                'planned_channels': DIRECTED_CHANNELS,
                'planned_queries': DIRECTED_SEARCH_QUERIES,
                'expanded_channels': expanded_channels,
                'directed_candidates_discovered': len(directed_pool),
                'directed_candidates_selected': len(directed_selection),
                'selection_path': DIRECTED_SELECTION_PATH,
            })
    finally:
        source_progress.close()
    added, existing = append_jsonl_once(DISCOVERED_PATH, discovered, id_field='video_id')
    write_json_atomic(DISCOVERY_FAILURES_PATH, discovery_failures)
    show_summary('Resumen del descubrimiento', {
        "sources_total": source_total,
        "sources_ok": source_outcomes['ok'],
        "sources_failed": source_outcomes['failed'],
        "candidates_unique": len(discovered),
        "candidates_added": added,
        "candidates_existing": existing,
        "expanded_channels": len(expanded_channels),
        "directed_cohort": len(directed_selection),
    }, tone='success' if not discovery_failures else 'warning')
    if discovery_failures:
        failure_counts = Counter(row['failure_kind'] for row in discovery_failures)
        show_summary('Fallos de fuentes por motivo', dict(sorted(failure_counts.items())), tone='warning')
        show_summary('Artefacto de auditoría', {'ruta': DISCOVERY_FAILURES_PATH}, tone='neutral')
else:
    show_callout('Descubrimiento desactivado', 'Se reutilizan candidatos y transcripciones locales.', tone='neutral')


## Cohorte activa y caché

In [ ]:
from moderacion_peru.acquisition import processed_video_ids

if DISCOVERY_MODE == 'directed':
    candidates = load_candidates(DIRECTED_SELECTION_PATH)
    candidate_origin = 'cohorte_dirigida_vigente'
elif DISCOVERY_MODE == 'both' and DISCOVER_NEW:
    seed_candidates_current = [
        candidate for candidate in discovered
        if candidate.get('sampling_mode') == 'seed'
    ]
    candidates = merge_candidates(seed_candidates_current, directed_selection)
    candidate_origin = 'descubrimiento_actual_seed_más_cohorte_dirigida'
else:
    candidate_files = [
        ROOT/'datos/raw/video_candidates.jsonl',
        ROOT/'datos/raw/videos_candidatos.csv',
    ]
    candidates = merge_candidates(*(load_candidates(source) for source in candidate_files))
    candidate_origin = 'archivo_acumulado_general'

processed_ids = processed_video_ids(CANONICAL)
existing_candidate_ids = {str(candidate['video_id']).strip() for candidate in candidates} & processed_ids
pending_candidates = [
    candidate for candidate in candidates
    if str(candidate['video_id']).strip() not in processed_ids
]
cached_ids = {path.stem for path in CACHE.glob('*.json')}
pending_cached = sum(
    str(candidate['video_id']).strip() in cached_ids for candidate in pending_candidates
)
show_summary('Candidatos filtrados antes de la adquisición', {
    'origen': candidate_origin,
    'únicos': len(candidates),
    'ya_canónicos_omitidos': len(existing_candidate_ids),
    'pendientes_totales': len(pending_candidates),
    'pendientes_reutilizables_desde_caché': pending_cached,
    'pendientes_que_requieren_red': len(pending_candidates) - pending_cached,
}, tone='neutral')


## Ejecución controlada y tolerante a fallos

In [ ]:
from functools import partial
from tqdm.auto import tqdm
from moderacion_peru.acquisition import fetch_youtube_subtitles, ingest_incremental

FAILURES = ROOT/'datos/raw/fallos_adquisicion.jsonl'
fetcher = partial(
    fetch_youtube_subtitles,
    languages=SUBTITLE_LANGUAGES,
    retries=YT_RETRIES,
    sleep_min_seconds=YT_SLEEP_MIN_SECONDS,
    sleep_max_seconds=YT_SLEEP_MAX_SECONDS,
)
if pending_candidates:
    video_progress = tqdm(total=len(pending_candidates), desc='Procesando pendientes', unit='video')

    def report_acquisition(event):
        counters = event['counters']
        video_progress.update(1)
        video_progress.set_postfix(
            existentes=counters['already_canonical'],
            cache=counters['reused_cache'],
            nuevos=counters['fetched'],
            fallidos=counters['failed'],
            diferidos=counters['deferred_by_limit'],
            pausa_429=counters['deferred_rate_limit'],
        )

    try:
        stats = ingest_incremental(
            pending_candidates,
            CANONICAL,
            CACHE,
            fetcher=fetcher if FETCH_NEW else None,
            failure_path=FAILURES,
            max_new_videos=MAX_NEW_VIDEOS,
            stop_on_error=STOP_ON_VIDEO_ERROR,
            progress_callback=report_acquisition,
        )
    finally:
        video_progress.close()
    stats = {
        'candidates_total': len(candidates),
        'filtered_existing_before_run': len(existing_candidate_ids),
        'pending_before_run': len(pending_candidates),
        **stats,
    }
    show_summary('Resumen de adquisición', stats, tone='success' if not stats['failed'] else 'warning')
    if stats['failed']:
        show_summary('Fallos de adquisición por motivo', {
            key.removeprefix('failure_'): value
            for key, value in stats.items()
            if key.startswith('failure_') and not key.startswith('failure_records_') and value
        }, tone='warning')
        show_summary('Videos omitidos sin detener el lote', {
            'cantidad': stats['failed'],
            'detalle': FAILURES,
        }, tone='warning')
elif candidates:
    show_callout(
        'Sin videos pendientes',
        'Todos los candidatos ya tienen una transcripción canónica; no se iniciaron descargas.',
        tone='success',
    )
else:
    show_callout(
        'No hay candidatos',
        'Active DISCOVER_NEW o añada un CSV/JSONL. El corpus existente no se vuelve a descargar.',
        tone='warning',
    )


## Referencias

[1] yt-dlp contributors, "yt-dlp: A Feature-Rich Command-Line Audio/Video Downloader," GitHub repository, 2026. [Online]. Available: https://github.com/yt-dlp/yt-dlp. Accessed: Aug. 5, 2026.

[2] R. Tatman, "Gender and Dialect Bias in YouTube's Automatic Captions," in Proc. 1st ACL Workshop Ethics NLP, 2017, pp. 53–59, doi: 10.18653/v1/W17-1606.

[3] YouTube, "Terms of Service," Nov. 2023. [Online]. Available: https://www.youtube.com/t/terms. Accessed: Aug. 5, 2026.

[4] A. S. franzke, A. Bechmann, M. Zimmer, et al., "Internet Research: Ethical Guidelines 3.0," Association of Internet Researchers, 2020. [Online]. Available: https://aoir.org/reports/ethics3.pdf

[5] Y. Fairstein, O. Kalinsky, Z. Karnin, et al., "Class Balancing for Efficient Active Learning in Imbalanced Datasets," in Proc. 18th Linguistic Annotation Workshop, 2024, pp. 77–86, doi: 10.18653/v1/2024.law-1.8.

[6] Y. Huang, B. Giledereli, A. Köksal, et al., "Balancing Methods for Multi-label Text Classification with Long-Tailed Class Distribution," in Proc. EMNLP, 2021, pp. 8153–8161, doi: 10.18653/v1/2021.emnlp-main.643.